# Download Visual Basemaps from Planet API

This notebook downloads monthly satellite imagery from Planet's global mosaics API, focusing on Santa Barbara County, California for the period 2019-2023.

In [2]:
import os
import sys
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from tqdm import tqdm
from datetime import datetime

sys.path.append("../utils")

import config
import data_utils
import planet_api
import planet_utils

### Authenticate with Planet API

In [3]:
PLANET_API_KEY = planet_api.key

planet = planet_utils.PlanetBasemapsAPI(api_key=PLANET_API_KEY)
print(f"Authentication status: {planet.check_authentication_status()}")

Authentication status: Authentication successful.


### Get montly mosaics from 2019-2023

In [4]:
# Get list of gliobal monthly mosaics
mosaics = planet.get_mosaics(search_str="global_monthly")
mosaics

['global_monthly_2016_01_mosaic',
 'global_monthly_2016_02_mosaic',
 'global_monthly_2016_03_mosaic',
 'global_monthly_2016_04_mosaic',
 'global_monthly_2016_05_mosaic',
 'global_monthly_2016_06_mosaic',
 'global_monthly_2016_07_mosaic',
 'global_monthly_2016_08_mosaic',
 'global_monthly_2016_09_mosaic',
 'global_monthly_2016_10_mosaic',
 'global_monthly_2016_11_mosaic',
 'global_monthly_2016_12_mosaic',
 'global_monthly_2017_01_mutate_mosaic',
 'global_monthly_2017_02_mosaic',
 'global_monthly_2017_03_mosaic',
 'global_monthly_2017_04_mosaic',
 'global_monthly_2017_05_mosaic',
 'global_monthly_2017_06_mosaic',
 'global_monthly_2017_07_mosaic',
 'global_monthly_2017_08_mosaic',
 'global_monthly_2017_09_mosaic',
 'global_monthly_2017_10_mosaic',
 'global_monthly_2017_11_mosaic',
 'global_monthly_2017_12_mosaic',
 'global_monthly_2018_01_mosaic',
 'global_monthly_2018_02_mosaic',
 'global_monthly_2018_03_mosaic',
 'global_monthly_2018_04_mosaic',
 'global_monthly_2018_05_mosaic',
 'globa

In [5]:
# Define time range for imagery
start_date = datetime(2019, 1, 1)
end_date = datetime(2023, 12, 31)  # Include all of January 2024

# Helper function to parse year and month from mosaic name
def extract_date(mosaic_name: str) -> datetime:
    """Extract datetime object from a mosaic name string."""
    parts = mosaic_name.split('_')
    year = int(parts[2])
    month = int(parts[3])
    return datetime(year, month, 1)

# Filter mosaics by date range
mosaics = [
    mosaic for mosaic in mosaics
    if start_date <= extract_date(mosaic) <= end_date
]
mosaics

['global_monthly_2019_01_mosaic',
 'global_monthly_2019_02_mosaic',
 'global_monthly_2019_03_mosaic',
 'global_monthly_2019_04_mosaic',
 'global_monthly_2019_05_mosaic',
 'global_monthly_2019_06_mosaic',
 'global_monthly_2019_07_mosaic',
 'global_monthly_2019_08_mosaic',
 'global_monthly_2019_09_mosaic',
 'global_monthly_2019_10_mosaic',
 'global_monthly_2019_11_mosaic',
 'global_monthly_2019_12_mosaic',
 'global_monthly_2020_01_mosaic',
 'global_monthly_2020_02_mosaic',
 'global_monthly_2020_03_mosaic',
 'global_monthly_2020_04_mosaic',
 'global_monthly_2020_05_mosaic',
 'global_monthly_2020_06_mosaic',
 'global_monthly_2020_07_mosaic',
 'global_monthly_2020_08_mosaic',
 'global_monthly_2020_09_mosaic',
 'global_monthly_2020_10_mosaic',
 'global_monthly_2020_11_mosaic',
 'global_monthly_2020_12_mosaic',
 'global_monthly_2021_01_mosaic',
 'global_monthly_2021_02_mosaic',
 'global_monthly_2021_03_mosaic',
 'global_monthly_2021_04_mosaic',
 'global_monthly_2021_05_mosaic',
 'global_month

### Download Each Mosaic for Santa Barbara County

In [7]:
for mosaic in mosaics:
    print(mosaic)

    # Set current mosaic
    planet.set_mosaic(mosaic)

    # Create directories for storing data in `/data/wildfire_prep/basemaps`
    mosaic_dir = os.path.join(
        config.basemap_dir,
        mosaic,
    )

    file_dir = os.path.join(mosaic_dir, "quad_ids")
    download_dir = os.path.join(mosaic_dir, "basemap_quads")

    os.makedirs(file_dir, exist_ok=True)
    os.makedirs(download_dir, exist_ok=True)

    quad_fn = os.path.join(file_dir, "quad_ids.geojson")

    counties = gpd.read_file(
        os.path.join(config.data_dir, "ca_counties", "CA_Counties.shp")
    )
    counties = counties.to_crs(config.geodetic_crs)
    sb_county = counties.loc[counties["NAME"] == "Santa Barbara"]
    bbox = [-125, 34.25, -119.0, 38.0]
    sb_county = sb_county.clip(bbox)

    bbox_aoi = sb_county.geometry.total_bounds

    quads = planet.get_items(bbox_aoi)

    quad_df = planet.convert_items_to_geodataframe(quads)
    quad_df = quad_df.drop_duplicates(subset="id")

    quad_df = quad_df.sjoin(sb_county, predicate="intersects")
    quad_df.drop(columns=["index_right"], inplace=True)

    filtered_ids = set(quad_df["id"].unique())

    if not os.path.exists(quad_fn):
        quad_df.to_file(quad_fn, driver="GeoJSON")

    quads = [q for q in quads if q["id"] in filtered_ids]

    planet.download_quads(
        quads,
        directory=download_dir,
        overwrite=False, # This prevents existing files from being redownloaded!
        log_interval=10,
    )

global_monthly_2019_01_mosaic
2025-05-07 06:44:22,829 - INFO - Download complete. Total: 0 downloaded, 43 skipped
global_monthly_2019_02_mosaic
2025-05-07 06:44:23,355 - INFO - Download complete. Total: 0 downloaded, 43 skipped
global_monthly_2019_03_mosaic
2025-05-07 06:44:23,877 - INFO - Download complete. Total: 0 downloaded, 43 skipped
global_monthly_2019_04_mosaic
2025-05-07 06:44:24,333 - INFO - Download complete. Total: 0 downloaded, 43 skipped
global_monthly_2019_05_mosaic
2025-05-07 06:44:24,755 - INFO - Download complete. Total: 0 downloaded, 43 skipped
global_monthly_2019_06_mosaic
2025-05-07 06:44:25,225 - INFO - Download complete. Total: 0 downloaded, 43 skipped
global_monthly_2019_07_mosaic
2025-05-07 06:44:25,633 - INFO - Download complete. Total: 0 downloaded, 43 skipped
global_monthly_2019_08_mosaic
2025-05-07 06:44:26,053 - INFO - Download complete. Total: 0 downloaded, 43 skipped
global_monthly_2019_09_mosaic
2025-05-07 06:44:26,443 - INFO - Download complete. Total:

In [ ]:
# for mosaic in mosaics:
#     print(mosaic)
    
#     # mosaic = mosaics[-1]

#     # Set current mosaic
#     planet.set_mosaic(mosaic)

#     # Create directories for storing data in `/data/wildfire_prep/basemaps`
#     mosaic_dir = os.path.join(
#         config.basemap_dir,
#         mosaic,
#     )

#     file_dir = os.path.join(mosaic_dir, "quad_ids")
#     download_dir = os.path.join(mosaic_dir, "basemap_quads")

#     os.makedirs(file_dir, exist_ok=True)
#     os.makedirs(download_dir, exist_ok=True)

#     quad_fn = os.path.join(file_dir, "quad_ids.geojson")
#     # grid_fn = os.path.join(file_dir, "grid_coordinates.parquet")

#     counties = gpd.read_file(
#         os.path.join(config.data_dir, "ca_counties", "CA_Counties.shp")
#     )
#     counties = counties.to_crs(config.geodetic_crs)
#     sb_county = counties.loc[counties["NAME"] == "Santa Barbara"]
#     bbox = [-125, 34.25, -119.0, 38.0]
#     sb_county = sb_county.clip(bbox)

#     bbox_aoi = sb_county.geometry.total_bounds

#     quads = planet.get_items(bbox_aoi)

#     quad_df = planet.convert_items_to_geodataframe(quads)
#     quad_df = quad_df.drop_duplicates(subset="id")

#     quad_df = quad_df.sjoin(sb_county, predicate="intersects")
#     quad_df.drop(columns=["index_right"], inplace=True)

#     filtered_ids = set(quad_df["id"].unique())

#     if not os.path.exists(quad_fn):
#         quad_df.to_file(quad_fn, driver="GeoJSON")

#     quads = [q for q in quads if q["id"] in filtered_ids]

#     planet.download_quads(
#         quads,
#         directory=download_dir,
#         overwrite=False,
#         log_interval=10,
#     )